# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import json

# Save a Plotly mimebundle (renders in Jupyter) without embedding plotly.js.
pio.renderers.default = "plotly_mimetype"


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


### Task 1 — Design decisions

**Chart type:** `px.choropleth`, using Plotly's built-in ISO-3 country shapes. A world comparison should be one static projection you can read at a glance. A tile basemap (`px.choropleth_map`) is for zooming into a city or region, which is what Task 2 needs.

**Colour scale:** diverging `RdBu`, centred at 0. The number on the map is years above or below the world mean, so the middle of the scale is a real value (the average), not an arbitrary colour stop. Red is below average, blue is above. A sequential scale would only say "more" or "less" and would hide which side of the average a country is on.

**What "region" means here:** Gapminder's `continent`. The title names the continent whose countries sit furthest below the unweighted mean `gm_2007['lifeExp'].mean()`. The single country furthest below that mean is also printed below; it falls inside that continent.


In [2]:
# Task 1 — life expectancy in 2007 relative to the world mean.
# The mean is the unweighted mean of Gapminder countries, as specified.

gm = px.data.gapminder()
gm_2007 = gm.loc[gm['year'] == 2007].copy()
world_mean = float(gm_2007['lifeExp'].mean())
gm_2007['lifeExp_vs_mean'] = gm_2007['lifeExp'] - world_mean

region_gap = (
    gm_2007.groupby('continent')['lifeExp_vs_mean']
    .mean()
    .sort_values()
)
furthest_region = region_gap.index[0]
furthest_gap = float(region_gap.iloc[0])
lowest_country = gm_2007.loc[gm_2007['lifeExp_vs_mean'].idxmin()]

print(f"World mean life expectancy (unweighted, 2007): {world_mean:.1f} years")
print("Mean deviation by continent (years):")
print(region_gap.round(2).to_string())
print(
    f"Furthest region below average: {furthest_region} ({furthest_gap:+.1f} years). "
    f"Lowest country: {lowest_country['country']} "
    f"({lowest_country['lifeExp']:.1f}, {lowest_country['lifeExp_vs_mean']:+.1f})."
)

fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp_vs_mean',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    hover_name='country',
    hover_data={
        'country': True,
        'lifeExp': ':.1f',
        'lifeExp_vs_mean': ':+.1f',
        'iso_alpha': False,
        'continent': False,
    },
    labels={
        'country': 'Country',
        'lifeExp': 'Life expectancy (years)',
        'lifeExp_vs_mean': 'Years vs world mean',
    },
    title=(
        f"{furthest_region} is furthest below the 2007 world average "
        f"life expectancy ({furthest_gap:.0f} years)"
    ),
    subtitle=(
        f"Deviation from the unweighted country mean ({world_mean:.1f} years). "
        "Red = below average, blue = above. Scale centred at 0."
    ),
    projection='natural earth',
    template='plotly_white',
    height=560,
)

fig.update_layout(
    font=dict(family='Arial', size=13),
    geo=dict(showframe=False, showcountries=True, bgcolor='white'),
    coloraxis_colorbar=dict(title='Years vs mean', thickness=14, len=0.75),
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()


World mean life expectancy (unweighted, 2007): 67.0 years
Mean deviation by continent (years):
continent
Africa     -12.20
Asia         3.72
Americas     6.60
Europe      10.64
Oceania     13.71
Furthest region below average: Africa (-12.2 years). Lowest country: Swaziland (39.6, -27.4).


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** London borough boundaries from the MIT-licensed file [`london_boroughs.geojson`](https://github.com/radoi90/housequest-data/blob/master/london_boroughs.geojson) (Emil Culic, 2014; see `data/london_boroughs.LICENSE.txt`). A simplified copy is stored at `data/london_boroughs.geojson` (Douglas–Peucker tolerance 0.0005°) so the notebook runs without a download. The feature property used for the join is `name`.

**Data:** `data/airbnb_london.csv` (this course's Airbnb London extract). The numeric variable is **median nightly price (£)** per borough. The extract covers 10 inner boroughs, not all 33. One name does not match the boundary file and is mapped before the join: `Hammersmith` → `Hammersmith and Fulham`. `Westminster` already matches. Median is used, rather than the mean, so a handful of extreme nightly prices do not set the borough colour.

**Chart type chosen** (`px.choropleth_map`) **and reason:**

`px.choropleth_map`. In Plotly 7 this is the tile-basemap choropleth (`px.choropleth_mapbox` is not in this version; it was renamed). Boroughs only make sense next to the river and the neighbouring polygons, so the map needs pan and zoom. `px.choropleth` is a static world projection — the right choice for Task 1, the wrong frame for one city. Basemap style is `carto-positron` so the price colours stay readable.

**Colour scale chosen** (sequential) **and reason:**

Sequential `YlOrRd`. Median nightly price is a magnitude: every borough is positive, and a higher value means a dearer borough. There is no signed deviation from a meaningful centre on this map, so a diverging scale would invent a midpoint. Pale yellow is cheaper, dark red is dearer.


In [3]:
# Task 2 — median nightly price across the inner London boroughs in the extract.

with open('../data/london_boroughs.geojson', 'r') as file:
    london_geojson = json.load(file)

# The property that matches the dataframe is `name`.
print(london_geojson['features'][0]['properties'])

airbnb = pd.read_csv('../data/airbnb_london.csv')

# The course file shortens one official borough name.
name_in_geojson = {'Hammersmith': 'Hammersmith and Fulham'}

borough = (
    airbnb.groupby('neighbourhood', as_index=False)['price']
    .median()
    .rename(columns={'price': 'median_price'})
)
borough['name'] = borough['neighbourhood'].replace(name_in_geojson)
borough = borough.sort_values('median_price', ascending=False)

geo_names = {feat['properties']['name'] for feat in london_geojson['features']}
unmatched = sorted(set(borough['name']) - geo_names)
print(f"GeoJSON boroughs: {len(geo_names)}")
print(f"Boroughs in the Airbnb extract: {len(borough)}")
print("Unmatched after the name map:", unmatched or "none")
print(borough[['name', 'median_price']].round(1).to_string(index=False))

dearest = borough.iloc[0]['name']
second = borough.iloc[1]['name']

fig = px.choropleth_map(
    borough,
    geojson=london_geojson,
    locations='name',
    featureidkey='properties.name',
    color='median_price',
    color_continuous_scale='YlOrRd',
    hover_name='name',
    hover_data={
        'name': False,
        'neighbourhood': False,
        'median_price': ':.0f',
    },
    labels={'median_price': 'Median nightly price (£)'},
    title=(
        f"Central-west London costs the most: {dearest} and {second} "
        f"have the highest median nightly prices"
    ),
    subtitle=(
        "Median nightly price in the course Airbnb extract "
        "(10 inner boroughs). Pale = cheaper, dark red = dearer."
    ),
    map_style='carto-positron',
    center={'lat': 51.49, 'lon': -0.12},
    zoom=10,
    opacity=0.85,
    height=680,
)

fig.update_layout(
    font=dict(family='Arial', size=13),
    coloraxis_colorbar=dict(title='Median £ / night', thickness=14, len=0.7),
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()


{'name': 'Barking and Dagenham'}
GeoJSON boroughs: 33
Boroughs in the Airbnb extract: 10
Unmatched after the name map: none
                  name  median_price
           Westminster         161.3
Kensington and Chelsea         161.0
Hammersmith and Fulham         128.2
                Camden         126.0
             Southwark         113.8
             Islington         109.3
            Wandsworth         104.6
               Hackney         103.3
               Lambeth          99.1
         Tower Hamlets          98.0
